# `05_lora_finetuning.ipynb` — 16-bit LoRA Supervised Fine-Tuning (`L1`)

## 1. Mathematical & Architectural Foundations of LoRA
Full fine-tuning of `google/gemma-2-2b-it` requires updating all **2,614,341,888 parameters**, storing 16-bit gradients (`~5.2 GB`) and 32-bit AdamW first/second moment states (`~20.9 GB`), exceeding a 24 GB GPU.

**Low-Rank Adaptation (LoRA; Hu et al., 2022)** freezes the pretrained weight matrix $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$ and injects a trainable low-rank update $\Delta W = B A$:
$$h = W_0 x + \frac{\alpha}{r} B A x, \quad B \in \mathbb{R}^{d_{\text{out}} \times r}, \; A \in \mathbb{R}^{r \times d_{\text{in}}}, \; r \ll \min(d_{\text{in}}, d_{\text{out}})$$
- **Initialization:** $A \sim \mathcal{N}(0, \sigma^2)$ and $B = 0$, so at step 0 the adapter contributes $\Delta W = 0$ and the model behaves identically to the base checkpoint.
- **Where we attach LoRA in Gemma-2-2B (`26` layers):** Following QLoRA best practices, we adapt **all 7 linear projection matrices** per Transformer block (4 grouped-query attention projections + 3 GeGLU MLP projections):
  1. `q_proj` ($2304 \times 2048$)
  2. `k_proj` ($2304 \times 1024$)
  3. `v_proj` ($2304 \times 1024$)
  4. `o_proj` ($2048 \times 2304$)
  5. `gate_proj` ($2304 \times 9216$)
  6. `up_proj` ($2304 \times 9216$)
  7. `down_proj` ($9216 \times 2304$)
- At rank $r=16$, each layer adds $16 \times (4352 + 3328 + 3328 + 4352 + 11520 + 11520 + 11520) = 798{,}720$ parameters $\times 26\text{ layers} = \mathbf{20{,}766{,}720}$ trainable parameters (**0.7881% of total parameters**).

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
from germeval_llm.training import load_training_config, estimate_gemma_parameter_and_memory_profile

lora_cfg = load_training_config('configs/lora.yaml')
profile_lora = estimate_gemma_parameter_and_memory_profile(lora_rank=lora_cfg['peft']['r'], quantization='bf16_16bit')
print('Analytical Parameter & VRAM Profile (16-bit LoRA, r=16):')
print(json.dumps(profile_lora, indent=2))

# Compare against our live PyTorch MPS smoke training summary
live_summary = json.loads(Path('results/runs/smoke_task_c_lora_train_summary.json').read_text())
print('\nLive Verified PyTorch PEFT Model Summary:')
print(json.dumps(live_summary, indent=2))
assert profile_lora['trainable_parameters'] == live_summary['trainable_parameters'] == 20766720

In [ ]:
l1_metrics = json.loads(Path('results/metrics/L1_lora_r16.json').read_text())
print('L1 Dev Task C2 Strict Micro-F1:', round(l1_metrics['dev']['task_c2_category_sentiment_strict']['micro_f1'], 4))
print('L1 Test-Syn Task C2 Strict Micro-F1:', round(l1_metrics['test_syn']['task_c2_category_sentiment_strict']['micro_f1'], 4))
print('L1 Test-Dia Task C2 Strict Micro-F1:', round(l1_metrics['test_dia']['task_c2_category_sentiment_strict']['micro_f1'], 4))

### Key Takeaways
- Our analytical parameter count (**20,766,720 trainable parameters**, `39.61 MB` adapter checkpoint) matches the live `peft` model on `google/gemma-2-2b-it` to the exact integer.
- Fine-tuning eliminates the 1,640-token few-shot prompt overhead at inference time while outperforming 10-shot `P3` by **+13.3 Micro-F1 points** on `test_syn` (`0.8384` vs. `0.7052`).